In [ ]:
import os
import pandas as pd
import json
from sqlalchemy.exc import IntegrityError
from db.connection import SessionLocal
from db.models import AppointmentLoad, DoctorDF,AgeSegment,DepartmentSegment,TestSegment

# Folder containing CSV files
CSV_FOLDER = "CSV_Data"

# Map CSV filenames (without extension) to models
CSV_MODEL_MAP = {
    "unpivoted_appointment_load": AppointmentLoad,
    "doctors": DoctorDF,
    "age_group_segment": AgeSegment,
    "dept_segment": DepartmentSegment,
    "test_segment":TestSegment
}

# Column name normalization mapping if needed
COLUMN_MAPPING = {
    "Doctor ID": "doctor_id",
    "Name": "name",
    "Gender": "gender",
    "Department": "department",
    "Available Days": "available_days",
    "Available Slots": "available_slots"
}

session = SessionLocal()

for csv_file in os.listdir(CSV_FOLDER):
    if not csv_file.endswith(".csv"):
        continue

    csv_name = os.path.splitext(csv_file)[0]
    model = CSV_MODEL_MAP.get(csv_name)
    if not model:
        print(f"Skipping {csv_file}: No corresponding model found.")
        continue

    file_path = os.path.join(CSV_FOLDER, csv_file)
    df = pd.read_csv(file_path)

    # Normalize column names to match model
    df.rename(columns=lambda x: COLUMN_MAPPING.get(x, x.lower()), inplace=True)

    print(f"Migrating {csv_file} -> {model.__name__} ({len(df)} rows)")

    try:
        for _, row in df.iterrows():
            data = row.to_dict()

            # Parse JSON columns if any
            for col in model.__table__.columns:
                if str(col.type) == "JSON" and col.name in data:
                    val = data[col.name]
                    if pd.isna(val):
                        data[col.name] = None
                    elif isinstance(val, str):
                        try:
                            data[col.name] = json.loads(val)
                        except:
                            pass

            instance = model(**data)
            session.add(instance)

        session.commit()
        print(f"{csv_file} migrated successfully.")

    except IntegrityError as e:
        print(f"Error committing {csv_file}: {e}")
        session.rollback()

session.close()
print("Migration finished.")


Skipping age_group_segment.csv: No corresponding model found.
Migrating dept_segment.csv -> DepartmentSegment (2 rows)
dept_segment.csv migrated successfully.
Skipping disorder_segment.csv: No corresponding model found.
Skipping doctor_disorder_segment.csv: No corresponding model found.
Migrating test_segment.csv -> TestSegment (230 rows)
test_segment.csv migrated successfully.
Migration finished.
